In [ ]:
%pip install -q vllm sentence-transformers tqdm

In [ ]:
import os
os.environ['USE_TF'] = '0'
os.environ['USE_FLAX'] = '0'
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TRANSFORMERS_NO_FLAX'] = '1'

import json
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.preprocessing import LabelEncoder
from sklearn.neighbors import NearestNeighbors
from sentence_transformers import SentenceTransformer
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import warnings
warnings.filterwarnings('ignore')

# ===== Константы =====
DATA_DIR = Path('../../data/children_products')
CLEANED_CSV = DATA_DIR / 'clildren_product_cleaned.csv'
RAW_CSV = DATA_DIR / '!03&04_17_VSE.csv'

MIN_INTERACTIONS = 5
TRAIN_SPLIT = 0.8

N_EVAL_USERS = 300
K_RECOMMEND = 30
TOP_M_PER_QUERY = 3
FINAL_K = 20
MAX_HISTORY_ITEMS = 30

EMBED_MODEL = 'sentence-transformers/paraphrase-multilingual-mpnet-base-v2'
SKU_EMB_CACHE = DATA_DIR / 'sku_embeddings.npy'

# ===== Выбор модели =====
MODEL_CHOICE = 'qwen7b'  # 'qwen3b' | 'qwen7b' | 'mistral7b' | 'vikhr12b' | 'qwen14b' | 'qwen32b_awq'
MODEL_REGISTRY = {
    'qwen3b':      ('Qwen/Qwen2.5-3B-Instruct',                          'qwen3b',      None,  6,  'самая лёгкая'),
    'qwen7b':      ('Qwen/Qwen2.5-7B-Instruct',                          'qwen7b',      None,  14, 'дефолт, замена Llama-8B'),
    'mistral7b':   ('mistralai/Mistral-7B-Instruct-v0.3',                'mistral7b',   None,  14, 'не-Meta альтернатива'),
    'vikhr12b':    ('Vikhrmodels/Vikhr-Nemo-12B-Instruct-R-21-09-24',    'vikhr12b',    None,  24, 'дотренирован на русском'),
    'qwen14b':     ('Qwen/Qwen2.5-14B-Instruct',                         'qwen14b',     None,  28, 'покрепче, без квантизации'),
    'qwen32b_awq': ('Qwen/Qwen2.5-32B-Instruct-AWQ',                     'qwen32b_awq', 'awq', 17, 'INT4-квант 32B (~99% качества bf16)'),
}
LLM_MODEL, MODEL_SUFFIX, LLM_QUANT, _vram_est, _note = MODEL_REGISTRY[MODEL_CHOICE]
print(f'Using LLM: {LLM_MODEL}  (~{_vram_est} GB, {_note})')

LLM_CACHE = DATA_DIR / f'llm_recommendations_cache_vllm_{MODEL_SUFFIX}.json'

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

In [ ]:
df = pd.read_csv(CLEANED_CSV)
print(f'Исходный датасет: {df.shape}')

raw = pd.read_csv(
    RAW_CSV,
    sep=';', encoding='cp1251',
    usecols=['ID_SKU', 'Номенклатура', 'Группа2', 'Группа3', 'Тип'],
    dtype=str, low_memory=False
)
raw = raw.dropna(subset=['ID_SKU']).drop_duplicates('ID_SKU')

df = df.merge(raw[['ID_SKU', 'Номенклатура']], on='ID_SKU', how='left')
fallback = df['Группа3'].fillna('') + ' / ' + df['Тип'].fillna('')
df['Номенклатура'] = df['Номенклатура'].fillna(fallback)

df_filtered = df[(df['Статус'] == 'Доставлен') & (df['Отменено'] == 'Нет')].copy()
df_filtered = df_filtered.dropna(subset=['Телефон_new', 'ID_SKU', 'Дата'])
df_filtered['Дата'] = pd.to_datetime(df_filtered['Дата'], errors='coerce')
df_filtered = df_filtered.dropna(subset=['Дата'])

for _ in range(5):
    uc = df_filtered.groupby('Телефон_new').size()
    ic = df_filtered.groupby('ID_SKU').size()
    active_users = uc[uc >= MIN_INTERACTIONS].index
    active_items = ic[ic >= MIN_INTERACTIONS].index
    before = len(df_filtered)
    df_filtered = df_filtered[
        df_filtered['Телефон_new'].isin(active_users) &
        df_filtered['ID_SKU'].isin(active_items)
    ]
    if len(df_filtered) == before:
        break

user_enc, item_enc = LabelEncoder(), LabelEncoder()
df_filtered['user_id'] = user_enc.fit_transform(df_filtered['Телефон_new'])
df_filtered['item_id'] = item_enc.fit_transform(df_filtered['ID_SKU'])
n_users, n_items = len(user_enc.classes_), len(item_enc.classes_)
print(f'Пользователей: {n_users:,}  Товаров: {n_items:,}  Взаимодействий: {len(df_filtered):,}')

df_filtered = df_filtered.sort_values('Дата')
split_date = df_filtered['Дата'].quantile(TRAIN_SPLIT)
train_data = df_filtered[df_filtered['Дата'] < split_date].copy()
test_data = df_filtered[df_filtered['Дата'] >= split_date].copy()
print(f'Split date: {split_date}')
print(f'Train: {len(train_data):,}  Test: {len(test_data):,}')

train_int = train_data.groupby(['user_id', 'item_id']).size().reset_index(name='count')
test_int = test_data.groupby(['user_id', 'item_id']).size().reset_index(name='count')
train_matrix = csr_matrix(
    (train_int['count'].values, (train_int['user_id'].values, train_int['item_id'].values)),
    shape=(n_users, n_items)
)

test_user_items = test_int.groupby('user_id')['item_id'].apply(list).to_dict()
train_user_set = set(train_int['user_id'].unique())
warm_eval_users = [u for u in test_user_items if u in train_user_set]

rng = np.random.default_rng(RANDOM_SEED)
sample_users = rng.choice(warm_eval_users, size=min(N_EVAL_USERS, len(warm_eval_users)), replace=False)
sample_users = sorted(map(int, sample_users))
print(f'Подвыборка для LLM: {len(sample_users)} пользователей')

In [ ]:
item_name_map = (
    df_filtered.groupby('item_id')['Номенклатура']
    .agg(lambda s: s.mode().iloc[0] if len(s.mode()) else s.iloc[0])
)
item_names = [item_name_map.loc[i] for i in range(n_items)]

if SKU_EMB_CACHE.exists():
    sku_emb = np.load(SKU_EMB_CACHE)
    if sku_emb.shape[0] != n_items:
        sku_emb = None
else:
    sku_emb = None

encoder = SentenceTransformer(EMBED_MODEL)
if sku_emb is None:
    sku_emb = encoder.encode(item_names, batch_size=64, show_progress_bar=True, convert_to_numpy=True)
    np.save(SKU_EMB_CACHE, sku_emb)

knn = NearestNeighbors(metric='cosine', algorithm='brute').fit(sku_emb)
print(f'SKU embeddings: {sku_emb.shape}')

In [ ]:
train_sorted = train_data.sort_values('Дата')
user_history_cache = {}
for uid, g in train_sorted.groupby('user_id'):
    user_history_cache[int(uid)] = list(zip(g['Дата'].dt.date.astype(str), g['Номенклатура']))
user_train_items_cache = {
    int(uid): set(g['item_id'].tolist())
    for uid, g in train_sorted.groupby('user_id')
}

def build_user_history(user_id: int, max_items: int = MAX_HISTORY_ITEMS) -> str:
    items = user_history_cache.get(user_id, [])
    tail = items[-max_items:]
    return '\n'.join(f'- {date}: {name}' for date, name in tail)

In [ ]:
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

_eot_id = tokenizer.convert_tokens_to_ids('<|eot_id|>')
LLM_TERMINATORS = sorted({
    int(t) for t in [tokenizer.eos_token_id, _eot_id]
    if isinstance(t, int) and t is not None and t >= 0
})
print(f'Terminators: {LLM_TERMINATORS}')

_llm_kwargs = dict(
    model=LLM_MODEL,
    max_model_len=4096,
    gpu_memory_utilization=0.9,
    enforce_eager=False,
    trust_remote_code=False,
)
if LLM_QUANT is None:
    _llm_kwargs['dtype'] = 'bfloat16'
else:
    _llm_kwargs['quantization'] = LLM_QUANT
    _llm_kwargs['dtype'] = 'auto'

llm = LLM(**_llm_kwargs)
print(f'vLLM loaded: {LLM_MODEL}')

In [ ]:
SYSTEM_PROMPT = (
    'Ты — рекомендательная система детского интернет-магазина. '
    'На основе истории покупок пользователя предложи список новых товаров, '
    'которые он вероятно купит в ближайшее время. '
    'Ориентируйся на возраст ребёнка, предпочитаемые категории, бренды, ценовой сегмент. '
    'Не повторяй товары из истории. '
    'Отвечай СТРОГО в формате JSON: {"recommendations": ["название1", "название2", ...]}. '
    'Каждое название должно быть в формате "БРЕНД, ТИП, характеристики" (как в истории).'
)

def build_prompt(user_id: int, k: int = K_RECOMMEND) -> str:
    hist = build_user_history(user_id)
    user_prompt = (
        f'История покупок пользователя:\n{hist}\n\n'
        f'Предложи {k} новых товаров в JSON: '
        '{"recommendations": ["..."]}'
    )
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': user_prompt},
    ]
    return tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)


def _parse_json_array(raw_text: str) -> list:
    try:
        obj = json.loads(raw_text)
        if isinstance(obj, dict):
            for key in ('recommendations', 'items', 'products', 'list'):
                if key in obj and isinstance(obj[key], list):
                    return [str(x) for x in obj[key] if str(x).strip()]
            for v in obj.values():
                if isinstance(v, list):
                    return [str(x) for x in v if str(x).strip()]
        if isinstance(obj, list):
            return [str(x) for x in obj if str(x).strip()]
    except Exception:
        m = re.search(r'\[[\s\S]*\]', raw_text)
        if m:
            try:
                arr = json.loads(m.group(0))
                if isinstance(arr, list):
                    return [str(x) for x in arr if str(x).strip()]
            except Exception:
                pass
    return []

In [ ]:
# Загружаем кэш
llm_recs = {}
llm_raw = {}
if LLM_CACHE.exists():
    with open(LLM_CACHE, 'r', encoding='utf-8') as f:
        cache = json.load(f)
    llm_recs = {int(k): v for k, v in cache.get('recs', {}).items()}
    llm_raw = {int(k): v for k, v in cache.get('raw', {}).items()}
    print(f'Восстановлено из кэша: {len(llm_recs)} пользователей')

to_process = [u for u in sample_users if u not in llm_raw]
print(f'К обработке: {len(to_process)} пользователей')

if to_process:
    prompts = [build_prompt(uid) for uid in to_process]
    print(f'Средняя длина промпта: {np.mean([len(tokenizer.encode(p)) for p in prompts[:10]]):.0f} токенов')

    sampling_params = SamplingParams(
        temperature=0.0,
        max_tokens=1000,
        stop_token_ids=LLM_TERMINATORS,
    )

    t0 = time.time()
    outputs = llm.generate(prompts, sampling_params)
    elapsed = time.time() - t0

    total_gen = sum(len(o.outputs[0].token_ids) for o in outputs)
    print(f'\nBatch {len(to_process)} юзеров за {elapsed:.1f}s')
    print(f'  {elapsed / len(to_process):.2f}s/юзер')
    print(f'  {total_gen / elapsed:.0f} tokens/sec (суммарно по всему батчу)')

    for uid, out in zip(to_process, outputs):
        text = out.outputs[0].text
        llm_raw[uid] = _parse_json_array(text)

    with open(LLM_CACHE, 'w', encoding='utf-8') as f:
        json.dump(
            {'raw': {str(k): v for k, v in llm_raw.items()},
             'recs': {str(k): v for k, v in llm_recs.items()}},
            f, ensure_ascii=False
        )

non_empty_raw = sum(1 for v in llm_raw.values() if v)
print(f'\nНепустых JSON-ответов: {non_empty_raw}/{len(llm_raw)}')

In [ ]:
def llm_to_skus(
    llm_outputs: list,
    user_train_items: set,
    final_k: int = FINAL_K,
    m_per_query: int = TOP_M_PER_QUERY,
) -> list:
    if not llm_outputs:
        return []
    q_emb = encoder.encode(llm_outputs, batch_size=32, show_progress_bar=False, convert_to_numpy=True)
    _, idx = knn.kneighbors(q_emb, n_neighbors=m_per_query)
    seen, ranked = set(), []
    for row in idx:
        for cand in row:
            c = int(cand)
            if c in user_train_items or c in seen:
                continue
            seen.add(c)
            ranked.append(c)
            if len(ranked) >= final_k:
                return ranked
    return ranked


for uid in tqdm(sample_users, desc='retrieval'):
    raw_out = llm_raw.get(int(uid), [])
    llm_recs[int(uid)] = llm_to_skus(raw_out, user_train_items_cache.get(int(uid), set()))

with open(LLM_CACHE, 'w', encoding='utf-8') as f:
    json.dump(
        {'raw': {str(k): v for k, v in llm_raw.items()},
         'recs': {str(k): v for k, v in llm_recs.items()}},
        f, ensure_ascii=False
    )

non_empty = sum(1 for v in llm_recs.values() if v)
print(f'Непустых рекомендаций: {non_empty}/{len(llm_recs)}')

In [ ]:
def precision_at_k(recommended, relevant, k):
    rec_k = set(recommended[:k])
    return len(rec_k & set(relevant)) / len(rec_k) if rec_k else 0.0

def recall_at_k(recommended, relevant, k):
    rec_k = set(recommended[:k])
    return len(rec_k & set(relevant)) / len(set(relevant)) if relevant else 0.0

def map_at_k(recommended, relevant, k):
    relevant = set(relevant)
    if not relevant:
        return 0.0
    score, hits = 0.0, 0.0
    for i, item in enumerate(recommended[:k]):
        if item in relevant:
            hits += 1
            score += hits / (i + 1)
    return score / min(len(relevant), k)

def ndcg_at_k(recommended, relevant, k):
    relevant = set(relevant)
    if not relevant:
        return 0.0
    dcg = sum(1.0 / np.log2(i + 2) for i, item in enumerate(recommended[:k]) if item in relevant)
    idcg = sum(1.0 / np.log2(i + 2) for i in range(min(len(relevant), k)))
    return dcg / idcg if idcg > 0 else 0.0

K_VALUES = [5, 10, 20]

def aggregate(recs_by_user: dict, users: list) -> dict:
    out = {k: {m: [] for m in ['precision', 'recall', 'map', 'ndcg']} for k in K_VALUES}
    for u in users:
        rel = test_user_items.get(u, [])
        if not rel:
            continue
        rec = recs_by_user.get(u, [])
        for k in K_VALUES:
            out[k]['precision'].append(precision_at_k(rec, rel, k))
            out[k]['recall'].append(recall_at_k(rec, rel, k))
            out[k]['map'].append(map_at_k(rec, rel, k))
            out[k]['ndcg'].append(ndcg_at_k(rec, rel, k))
    return {k: {m: float(np.mean(v)) if v else 0.0 for m, v in metrics.items()} for k, metrics in out.items()}

llm_results = aggregate(llm_recs, sample_users)
print('LLM zero-shot (vLLM) метрики:')
for k in K_VALUES:
    print(f'  @{k}: ', {m: f'{v:.4f}' for m, v in llm_results[k].items()})

In [ ]:
try:
    from implicit.gpu.als import AlternatingLeastSquares
except ImportError:
    from implicit.als import AlternatingLeastSquares

als = AlternatingLeastSquares(
    factors=50, regularization=0.01, iterations=15,
    calculate_training_loss=True, random_state=42
)
als.fit(train_matrix)

als_recs = {}
for uid in tqdm(sample_users, desc='ALS recommend'):
    ids, _ = als.recommend(
        uid, train_matrix[uid],
        N=FINAL_K, filter_already_liked_items=True
    )
    als_recs[int(uid)] = [int(x) for x in ids]

als_results = aggregate(als_recs, sample_users)
print('ALS метрики:')
for k in K_VALUES:
    print(f'  @{k}: ', {m: f'{v:.4f}' for m, v in als_results[k].items()})

In [ ]:
rows = []
for k in K_VALUES:
    for m in ['precision', 'recall', 'map', 'ndcg']:
        rows.append({
            'metric': f'{m.upper()}@{k}',
            'LLM (vLLM)': llm_results[k][m],
            'ALS': als_results[k][m],
        })
comp_df = pd.DataFrame(rows)
comp_df['Δ %'] = (comp_df['LLM (vLLM)'] - comp_df['ALS']) / (comp_df['ALS'] + 1e-10) * 100

print(f"{'Метрика':<15} {'LLM':>10} {'ALS':>10} {'Δ':>10}")
print('-' * 50)
for _, r in comp_df.iterrows():
    print(f"{r['metric']:<15} {r['LLM (vLLM)']:>10.4f} {r['ALS']:>10.4f} {r['Δ %']:>+9.1f}%")
comp_df

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, metric in zip(axes.flat, ['precision', 'recall', 'map', 'ndcg']):
    llm_v = [llm_results[k][metric] for k in K_VALUES]
    als_v = [als_results[k][metric] for k in K_VALUES]
    ax.plot(K_VALUES, llm_v, marker='s', linewidth=2, label=f'LLM zero-shot ({MODEL_SUFFIX})')
    ax.plot(K_VALUES, als_v, marker='o', linewidth=2, label='ALS')
    ax.set_xlabel('K')
    ax.set_ylabel(f'{metric.upper()}@K')
    ax.set_title(f'{metric.upper()}@K')
    ax.legend()
    ax.grid(True, alpha=0.3)
plt.suptitle(f'LLM ({MODEL_SUFFIX}) vs ALS (n={len(sample_users)} пользователей)', fontsize=14)
plt.tight_layout()
plt.show()